# Bitcoin Sentiment Trader Analysis

Importing Library

In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

STEP 1: Load Datasets

In [44]:
sentiment = pd.read_csv('fear_greed_index.csv')
trades = pd.read_csv('historical_data.csv')

STEP 2: Data Cleaning

In [45]:
sentiment['date'] = pd.to_datetime(sentiment['date'])
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M')
trades['trade_date'] = trades['Timestamp IST'].dt.date
trades['trade_date'] = pd.to_datetime(trades['trade_date'])

STEP 3: Merge Datasets

In [46]:
merged = pd.merge(trades, sentiment[['date','classification','value']], left_on='trade_date', right_on='date', how='left')

STEP 4: Feature Engineering

In [47]:
merged['Net_PnL'] = merged['Closed PnL'] - merged['Fee']
merged['Profitability'] = np.where(
    merged['Net_PnL'] > 0,
    'Profit',
    'Loss'
)
merged['Trade_Value'] = merged['Execution Price'] * merged['Size Tokens']
merged['ROI'] = (merged['Net_PnL'] / merged['Trade_Value']) * 100
merged['Hour'] = merged['Timestamp IST'].dt.hour
merged['Day'] = merged['Timestamp IST'].dt.day_name()
merged.groupby('classification')['Net_PnL'].mean()
merged.groupby('classification')['Net_PnL'].median()
(merged['Net_PnL'] > 0)
merged.groupby('classification')['ROI'].mean()

,ROI
classification,
Extreme Fear,0.410868
Extreme Greed,3.976013
Fear,1.492224
Greed,1.938999
Neutral,0.951276


Analysis 1: Average PnL by Sentiment

In [48]:
avg_pnl = merged.groupby('classification')['Net_PnL'].mean().sort_values()
median_pnl = merged.groupby('classification')['Net_PnL'].median()
print('\nAverage PnL by Sentiment:\n', pnl_sentiment)


Average PnL by Sentiment:
 classification
Greed           -0.010485
Neutral         -0.007516
Extreme Fear    -0.005958
Fear            -0.005752
Extreme Greed   -0.001148
Name: Net_PnL, dtype: float64


Analysis 2: Win Rate by Sentiment

In [49]:
win_rate = merged.groupby('classification').apply(
    lambda x: (x['Net_PnL'] > 0).mean()*100
)
print('\nWin Rate by Sentiment:\n', win_rate)


Win Rate by Sentiment:
 classification
Extreme Fear     36.845794
Extreme Greed    46.769354
Fear             41.151738
Greed            39.124903
Neutral          39.590299
dtype: float64


/tmp/ipykernel_3778/4043219956.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  win_rate = merged.groupby('classification').apply(


Analysis 3: Trading Volume by Sentiment

In [50]:
volume_sentiment = merged.groupby('classification')['Size USD'].sum()
print('\nTrading Volume by Sentiment:\n', volume_sentiment)


Trading Volume by Sentiment:
 classification
Extreme Fear     1.144843e+08
Extreme Greed    1.244652e+08
Fear             4.833248e+08
Greed            2.885825e+08
Neutral          1.802421e+08
Name: Size USD, dtype: float64


Analysis 4: Best Performing Coins

In [51]:
coin_performance = merged.groupby(['classification','Coin'])['Net_PnL'].sum().reset_index()
print('\nTop Coin Performance:\n', coin_performance.sort_values('Net_PnL', ascending=False).head(10))


Top Coin Performance:
     classification  Coin       Net_PnL
88   Extreme Greed  @107  1.986856e+06
324           Fear  HYPE  8.289716e+05
361           Fear   SOL  7.300443e+05
387          Greed  @107  7.227457e+05
315           Fear   ETH  6.689009e+05
41    Extreme Fear  HYPE  4.779475e+05
539          Greed   SOL  4.467808e+05
307           Fear   BTC  4.249308e+05
472          Greed   ETH  3.429670e+05
694        Neutral   SOL  2.995284e+05


In [52]:
sharpe_ratio = merged.groupby('classification')['Net_PnL'].mean() / merged.groupby('classification')['Net_PnL'].std()
print(sharpe_ratio)

classification
Extreme Fear     0.029455
Extreme Greed    0.087690
Fear             0.056487
Greed            0.037172
Neutral          0.064503
Name: Net_PnL, dtype: float64


Hypothesis Testing (Fear vs Greed)

In [53]:
fear_pnl = merged[merged['classification'].str.contains('Fear', na=False)]['Net_PnL']
greed_pnl = merged[merged['classification'].str.contains('Greed', na=False)]['Net_PnL']

t_stat, p_value = ttest_ind(fear_pnl, greed_pnl, equal_var=False)
print('\nT-Test Result:')
print('T-statistic:', t_stat)
print('P-value:', p_value)


T-Test Result:
T-statistic: -1.0725354366317983
P-value: 0.28348111106908547


Visualizations

In [54]:
plt.figure(figsize=(10,6))
sns.barplot(x=avg_pnl.index, y=avg_pnl.values)
plt.xticks(rotation=45)
plt.title('Average PnL by Market Sentiment')
plt.tight_layout()
plt.savefig('avg_pnl_sentiment.png')
plt.close()

In [55]:
plt.figure(figsize=(10,6))
sns.barplot(x=win_rate.index, y=win_rate.values)
plt.xticks(rotation=45)
plt.title('Win Rate by Market Sentiment')
plt.tight_layout()
plt.savefig('win_rate_sentiment.png')
plt.close()

In [56]:
plt.figure(figsize=(10,6))
sns.boxplot(data=merged, x='classification', y='Net_PnL')
plt.xticks(rotation=45)
plt.title('PnL Distribution by Sentiment')
plt.tight_layout()
plt.savefig('pnl_distribution.png')
plt.close()

In [57]:
roi_analysis = merged.groupby('classification')['ROI'].mean()

plt.figure(figsize=(10,6))
sns.barplot(x=roi_analysis.index, y=roi_analysis.values)
plt.xticks(rotation=45)
plt.title('Average ROI by Sentiment')
plt.tight_layout()
plt.savefig('roi_sentiment.png')
plt.close()

Export final merged data

In [58]:
merged.to_csv('merged_analysis.csv', index=False)

print('\nAnalysis completed successfully!')


Analysis completed successfully!
